# E-Commerce Analytics
Sales, customer value, retention, product, geography, and operations analysis using the Olist dataset.

Observed CLV in this notebook means historical customer revenue; it is not a prediction of future value.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = next(p for p in (Path('../datasets'), Path('datasets')) if p.exists())
REPORT_DIR = Path('../sql/reports') if Path('../sql').exists() else Path('sql/reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
read = lambda name: pd.read_csv(DATA_DIR / name)
customers = read('olist_customers_dataset.csv')
orders = read('olist_orders_dataset.csv')
items = read('olist_order_items_dataset.csv')
products = read('olist_products_dataset.csv')
payments = read('olist_order_payments_dataset.csv')
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
for column in ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[column] = pd.to_datetime(orders[column])

In [ ]:
delivered = orders[orders.order_status.eq('delivered')].copy()
tx = (delivered[['order_id', 'customer_id', 'order_purchase_timestamp']]
      .merge(customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']], on='customer_id')
      .merge(items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], on='order_id'))
tx['revenue'] = tx['price'].astype(float)
tx['order_month'] = tx.order_purchase_timestamp.dt.to_period('M')
tx = tx.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
tx.head()

## RFM, Pareto, CLV, and churn risk

In [ ]:
as_of = tx.order_purchase_timestamp.max() + pd.Timedelta(days=1)
rfm = tx.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (as_of - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('revenue', 'sum'),
    state=('customer_state', 'first')
).reset_index()
for metric in ['recency', 'frequency', 'monetary']:
    rfm[metric + '_rank'] = rfm[metric].rank(method='first')
rfm['r_score'] = 6 - pd.qcut(rfm.recency_rank, 5, labels=False)
rfm['f_score'] = pd.qcut(rfm.frequency_rank, 5, labels=False) + 1
rfm['m_score'] = pd.qcut(rfm.monetary_rank, 5, labels=False) + 1
conditions = [(rfm.r_score >= 4) & (rfm.m_score >= 4), (rfm.r_score <= 2) & (rfm.m_score >= 4), (rfm.r_score >= 4) & (rfm.m_score <= 2), (rfm.r_score <= 2) & (rfm.m_score <= 2)]
rfm['segment'] = np.select(conditions, ['High Value Active', 'High Value At Risk', 'Recent Low Value', 'At Risk'], default='Regular Customers')
rfm['revenue_rank'] = rfm.monetary.rank(method='first', ascending=False).astype(int)
rfm.to_csv(REPORT_DIR / 'rfm_scores.csv', index=False)
rfm[['customer_unique_id', 'segment', 'monetary', 'state']].to_csv(REPORT_DIR / 'customer_segment.csv', index=False)
top_100 = rfm.nlargest(100, 'monetary')
top_100.to_csv(REPORT_DIR / 'top_100_customers.csv', index=False)
top_10_count = max(1, int(np.ceil(len(rfm) * 0.1)))
print('Top 10% revenue share:', f"{rfm.nsmallest(top_10_count, 'revenue_rank').monetary.sum() / rfm.monetary.sum():.1%}")
print('Average customer value:', round(rfm.monetary.mean(), 2))
rfm.head()

In [ ]:
clv = rfm.groupby('segment', as_index=False).agg(customers=('customer_unique_id', 'count'), average_clv=('monetary', 'mean'), total_value=('monetary', 'sum'))
clv.to_csv(REPORT_DIR / 'clv_by_segment.csv', index=False)
clv_by_state = rfm.groupby('state', as_index=False).agg(customers=('customer_unique_id', 'nunique'), total_revenue=('monetary', 'sum'), average_customer_value=('monetary', 'mean')).sort_values('average_customer_value', ascending=False)
clv_by_state.to_csv(REPORT_DIR / 'clv_by_state.csv', index=False)
clv_by_category = tx.groupby('product_category_name', dropna=False, as_index=False).agg(unique_customers=('customer_unique_id', 'nunique'), total_revenue=('revenue', 'sum')).sort_values('total_revenue', ascending=False)
clv_by_category['revenue_per_customer'] = clv_by_category.total_revenue / clv_by_category.unique_customers
clv_by_category.to_csv(REPORT_DIR / 'clv_by_category.csv', index=False)
dates = tx[['customer_unique_id', 'order_id', 'order_purchase_timestamp']].drop_duplicates().sort_values(['customer_unique_id', 'order_purchase_timestamp'])
dates['days_between_purchases'] = dates.groupby('customer_unique_id').order_purchase_timestamp.diff().dt.days
print('Average days between purchases:', round(dates.days_between_purchases.mean(), 1))
clv

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ordered_clv = rfm.sort_values('monetary', ascending=False).reset_index(drop=True)
ordered_clv['cumulative_revenue_percent'] = 100 * ordered_clv.monetary.cumsum() / ordered_clv.monetary.sum()
axes[0].plot((ordered_clv.index + 1) / len(ordered_clv) * 100, ordered_clv.cumulative_revenue_percent, color='darkorange')
axes[0].axvline(10, linestyle='--', color='grey')
axes[0].set(title='Customer Revenue Concentration', xlabel='Customers ranked by revenue (%)', ylabel='Cumulative revenue (%)')
sns.histplot(rfm.monetary, bins=40, ax=axes[1], color='steelblue')
axes[1].set(title='Observed Customer Value Distribution', xlabel='Lifetime revenue', ylabel='Customers')
plt.tight_layout()
plt.show()

state_clv = clv_by_state.head(15).set_index('state')[['average_customer_value']]
sns.heatmap(state_clv, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Average Observed CLV by State')
plt.show()

clv_by_category.head(15).plot.barh(x='product_category_name', y='revenue_per_customer', figsize=(10, 6), legend=False, color='teal')
plt.title('Category Revenue per Customer')
plt.xlabel('Revenue per unique customer')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
first = tx.groupby('customer_unique_id').order_purchase_timestamp.min().rename('cohort_month')
cohort = tx.join(first, on='customer_unique_id')
cohort['cohort_month'] = cohort.cohort_month.dt.to_period('M')
cohort['purchase_month'] = cohort.order_purchase_timestamp.dt.to_period('M')
cohort['months_since_first_purchase'] = (cohort.purchase_month - cohort.cohort_month).apply(lambda x: x.n)
retention = cohort.groupby(['cohort_month', 'months_since_first_purchase']).customer_unique_id.nunique().reset_index(name='active_customers')
sizes = retention[retention.months_since_first_purchase.eq(0)][['cohort_month', 'active_customers']].rename(columns={'active_customers': 'cohort_customers'})
retention = retention.merge(sizes, on='cohort_month')
retention['retention_rate'] = retention.active_customers / retention.cohort_customers
retention.to_csv(REPORT_DIR / 'cohort_retention.csv', index=False)
cohort_kpis = retention[retention.months_since_first_purchase.isin([0, 1, 3, 6])].pivot(index='cohort_month', columns='months_since_first_purchase', values='retention_rate')
cohort_kpis.columns = [f'month_{column}_retention' for column in cohort_kpis.columns]
cohort_kpis.reset_index().to_csv(REPORT_DIR / 'cohort_retention_kpis.csv', index=False)
sns.heatmap(retention.pivot(index='cohort_month', columns='months_since_first_purchase', values='retention_rate'), cmap='Blues')
plt.title('Cohort Retention')
plt.show()

In [ ]:
product_analysis = tx.groupby('product_category_name', dropna=False).agg(total_orders=('order_id', 'nunique'), total_revenue=('revenue', 'sum'), unique_products=('product_id', 'nunique'), avg_price=('price', 'mean')).reset_index().sort_values('total_revenue', ascending=False)
product_analysis['revenue_share_percent'] = 100 * product_analysis.total_revenue / product_analysis.total_revenue.sum()
product_analysis.to_csv(REPORT_DIR / 'product_analysis.csv', index=False)
time_series = tx.groupby('order_month').agg(total_orders=('order_id', 'nunique'), total_revenue=('revenue', 'sum')).reset_index()
time_series['avg_order_value'] = time_series.total_revenue / time_series.total_orders
time_series['revenue_growth_percent'] = time_series.total_revenue.pct_change() * 100
time_series.to_csv(REPORT_DIR / 'time_series_analysis.csv', index=False)
product_analysis.head(10)

In [ ]:
delivery = delivered[['order_id', 'customer_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']].copy()
delivery['delivery_days'] = (delivery.order_delivered_customer_date - delivery.order_purchase_timestamp).dt.total_seconds() / 86400
delivery['on_time'] = delivery.order_delivered_customer_date <= delivery.order_estimated_delivery_date
delivery['approval_days'] = (delivery.order_approved_at - delivery.order_purchase_timestamp).dt.total_seconds() / 86400
delivery['processing_days'] = (delivery.order_delivered_carrier_date - delivery.order_approved_at).dt.total_seconds() / 86400
delivery.to_csv(REPORT_DIR / 'delivery_performance.csv', index=False)
payment_summary = payments.groupby('payment_type', as_index=False).agg(orders=('order_id', 'nunique'), payment_value=('payment_value', 'sum'), average_installments=('payment_installments', 'mean'))
payment_summary.to_csv(REPORT_DIR / 'payment_analysis.csv', index=False)
delivery[['delivery_days', 'approval_days', 'processing_days']].describe()